# Candidate List PDF Parser and Seat Plan Mapper

This Jupyter Notebook extracts candidate table data from all DUET candidate list PDF files in the directory, maps each candidate to their Seat Plan room allocation (Date, Shift with Time, Building Name, and Room), combines them into a single pandas DataFrame, and exports the combined dataset to an Excel file.

### Requirements
Make sure you run this notebook inside the configured virtual environment (`.venv`) which has `pandas`, `openpyxl`, and `pdfplumber` installed.

In [1]:
import os
import glob
import re
import pdfplumber
import pandas as pd

# 1. Find all PDF files in the current directory
pdf_files = sorted(glob.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF candidate lists in the folder:")
for idx, file in enumerate(pdf_files, 1):
    print(f"{idx}. {file}")

Found 8 PDF candidate lists in the folder:
1. 1 Civil Engineering (CE).pdf
2. 2 Electrical and Electronic Engineering (EEE).pdf
3. 3 Mechanical Engineering (ME) IPE MME.pdf
4. 4 Computer Science and Engineering (CSE).pdf
5. 5 Textile Engineering (TE).pdf
6. 6 Architecture (Arch).pdf
7. 7 Chemical Engineering (ChE).pdf
8. 8 Food Engineering (FE).pdf


In [2]:
# 2. Parse all candidate list PDFs and combine into a list of DataFrames
all_dfs = []

for pdf_file in pdf_files:
    # Skip temporary or other PDF files if they don't match the candidate list format
    if not re.match(r'^\d+\s+', pdf_file):
        continue
        
    # Extract Department Name from filename (e.g. "1 Civil Engineering (CE).pdf" -> "Civil Engineering (CE)")
    dept_name = os.path.splitext(pdf_file)[0]
    dept_name = re.sub(r'^\d+\s*', '', dept_name).strip()
    
    print(f"Extracting tables from: '{pdf_file}' -> Department: '{dept_name}'")
    
    all_rows = []
    header = None
    
    with pdfplumber.open(pdf_file) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            if not tables:
                continue
            
            for table in tables:
                if not table:
                    continue
                
                # Establish header from the first page's first row
                if header is None:
                    header = [h.replace('\n', ' ').strip() if h else f"Col{idx}" for idx, h in enumerate(table[0])]
                    start_row_idx = 1
                else:
                    start_row_idx = 0
                
                for row in table[start_row_idx:]:
                    # Skip empty rows
                    if not any(row):
                        continue
                    # Skip repeated headers on later pages
                    if row[0] == 'SL' or 'Applicant' in str(row[1]):
                        continue
                    
                    # Adjust columns if row length is slightly off (padding or truncating)
                    if len(row) < len(header):
                        row = row + [''] * (len(header) - len(row))
                    elif len(row) > len(header):
                        row = row[:len(header)]
                        
                    cleaned_row = [str(cell).replace('\n', ' ').strip() if cell is not None else '' for cell in row]
                    all_rows.append(cleaned_row)
                    
    if all_rows:
        df = pd.DataFrame(all_rows, columns=header)
        df['Department'] = dept_name
        df['Source File'] = pdf_file
        all_dfs.append(df)
        print(f"  Successfully parsed {len(df)} candidates.")
    else:
        print(f"  Warning: No candidate rows parsed from '{pdf_file}'.")

# Combine all parsed tables into a single DataFrame
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\nCombined candidate database successfully built. Total Candidates: {len(combined_df)}")
else:
    print("\nNo data found to combine.")

Extracting tables from: '1 Civil Engineering (CE).pdf' -> Department: 'Civil Engineering (CE)'


  Successfully parsed 1647 candidates.
Extracting tables from: '2 Electrical and Electronic Engineering (EEE).pdf' -> Department: 'Electrical and Electronic Engineering (EEE)'


  Successfully parsed 1775 candidates.
Extracting tables from: '3 Mechanical Engineering (ME) IPE MME.pdf' -> Department: 'Mechanical Engineering (ME) IPE MME'


  Successfully parsed 1016 candidates.
Extracting tables from: '4 Computer Science and Engineering (CSE).pdf' -> Department: 'Computer Science and Engineering (CSE)'


  Successfully parsed 729 candidates.
Extracting tables from: '5 Textile Engineering (TE).pdf' -> Department: 'Textile Engineering (TE)'


  Successfully parsed 296 candidates.
Extracting tables from: '6 Architecture (Arch).pdf' -> Department: 'Architecture (Arch)'


  Successfully parsed 114 candidates.
Extracting tables from: '7 Chemical Engineering (ChE).pdf' -> Department: 'Chemical Engineering (ChE)'


  Successfully parsed 844 candidates.
Extracting tables from: '8 Food Engineering (FE).pdf' -> Department: 'Food Engineering (FE)'


  Successfully parsed 162 candidates.

Combined candidate database successfully built. Total Candidates: 6583


In [3]:
# 3. Set up the Seat Plan range allocations
dept_mapping = {
    "Civil Engineering (CE)": "CE",
    "Electrical and Electronic Engineering (EEE)": "EEE",
    "Mechanical Engineering (ME) IPE MME": "ME/IPE/MME",
    "Computer Science and Engineering (CSE)": "CSE",
    "Textile Engineering (TE)": "TE",
    "Architecture (Arch)": "Arch.",
    "Chemical Engineering (ChE)": "ChE",
    "Food Engineering (FE)": "FE"
}

slot_mapping = {
    "1st Shift": "1st Shift 09:30 AM to 12:00 PM",
    "2nd Shift": "2nd Shift 02:00 PM to 4:30 PM"
}

seat_plan = [
    # Building: Shahid Abu Sayed Administrative Building (SASAB)
    {"building": "Shahid Abu Sayed Administrative Building (SASAB)", "room": "11005 (10th Floor)", "dept": "CE", "start": 11297, "end": 11436, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Abu Sayed Administrative Building (SASAB)", "room": "11005 (10th Floor)", "dept": "ChE", "start": 70494, "end": 70633, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Abu Sayed Administrative Building (SASAB)", "room": "Seminar Room-2 (10th Floor)", "dept": "CE", "start": 11437, "end": 11542, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Abu Sayed Administrative Building (SASAB)", "room": "Seminar Room-2 (10th Floor)", "dept": "ChE", "start": 70634, "end": 70739, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Abu Sayed Administrative Building (SASAB)", "room": "Seminar Room (9th Floor)", "dept": "CE", "start": 11543, "end": 11647, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Abu Sayed Administrative Building (SASAB)", "room": "Seminar Room (9th Floor)", "dept": "ChE", "start": 70740, "end": 70844, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},

    # Building: Shahid Syed Nazrul Islam Academic Building (SSNIAB)
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "1st Floor Open Space", "dept": "CSE", "start": 40001, "end": 40040, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "1st Floor Open Space", "dept": "EEE", "start": 20001, "end": 20040, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2002", "dept": "CSE", "start": 40041, "end": 40096, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2002", "dept": "EEE", "start": 20041, "end": 20096, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2006", "dept": "CSE", "start": 40097, "end": 40136, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2006", "dept": "EEE", "start": 20097, "end": 20136, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2008", "dept": "CSE", "start": 40137, "end": 40176, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2008", "dept": "EEE", "start": 20137, "end": 20176, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2010", "dept": "CSE", "start": 40177, "end": 40216, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2010", "dept": "EEE", "start": 20177, "end": 20216, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2012", "dept": "CSE", "start": 40217, "end": 40272, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2012", "dept": "EEE", "start": 20217, "end": 20272, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2029", "dept": "ME/IPE/MME", "start": 30001, "end": 30056, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2029", "dept": "EEE", "start": 20273, "end": 20328, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2nd Floor Open Space", "dept": "ME/IPE/MME", "start": 30057, "end": 30096, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "2nd Floor Open Space", "dept": "EEE", "start": 20329, "end": 20368, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "3006", "dept": "ME/IPE/MME", "start": 30097, "end": 30136, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "3006", "dept": "EEE", "start": 20369, "end": 20408, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "3010", "dept": "ME/IPE/MME", "start": 30137, "end": 30176, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "3010", "dept": "EEE", "start": 20409, "end": 20448, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "3012", "dept": "ME/IPE/MME", "start": 30177, "end": 30232, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "3012", "dept": "EEE", "start": 20449, "end": 20504, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "3031", "dept": "ME/IPE/MME", "start": 30233, "end": 30288, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "3031", "dept": "EEE", "start": 20505, "end": 20560, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "4004", "dept": "ME/IPE/MME", "start": 30289, "end": 30318, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "4010", "dept": "ME/IPE/MME", "start": 30319, "end": 30348, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "5004", "dept": "ME/IPE/MME", "start": 30349, "end": 30383, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "5004", "dept": "FE", "start": 80001, "end": 80035, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "5005", "dept": "ME/IPE/MME", "start": 30384, "end": 30431, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "5005", "dept": "FE", "start": 80036, "end": 80083, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "5007", "dept": "ME/IPE/MME", "start": 30432, "end": 30461, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "6004-A", "dept": "ME/IPE/MME", "start": 30462, "end": 30517, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "6004-A", "dept": "EEE", "start": 20561, "end": 20616, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "6005", "dept": "ME/IPE/MME", "start": 30518, "end": 30557, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "6005", "dept": "EEE", "start": 20617, "end": 20656, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "6006-B", "dept": "ME/IPE/MME", "start": 30558, "end": 30597, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "6006-B", "dept": "EEE", "start": 20657, "end": 20696, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "6017", "dept": "ME/IPE/MME", "start": 30598, "end": 30657, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "6017", "dept": "EEE", "start": 20697, "end": 20756, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "6th Floor Open Space", "dept": "ME/IPE/MME", "start": 30658, "end": 30697, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "6th Floor Open Space", "dept": "EEE", "start": 20757, "end": 20796, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "7019", "dept": "ME/IPE/MME", "start": 30698, "end": 30775, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "7019", "dept": "EEE", "start": 20797, "end": 20874, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "7026", "dept": "ME/IPE/MME", "start": 30776, "end": 30837, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "7026", "dept": "EEE", "start": 20875, "end": 20936, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "7th Floor Open Space", "dept": "ME/IPE/MME", "start": 30838, "end": 30877, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "7th Floor Open Space", "dept": "EEE", "start": 20937, "end": 20976, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "8009", "dept": "ME/IPE/MME", "start": 30878, "end": 30937, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "8009", "dept": "EEE", "start": 20977, "end": 21036, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "8010", "dept": "ME/IPE/MME", "start": 30938, "end": 31016, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "8010", "dept": "FE", "start": 80084, "end": 80162, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "8020", "dept": "CSE", "start": 40273, "end": 40332, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "8020", "dept": "EEE", "start": 21037, "end": 21096, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "8026", "dept": "CSE", "start": 40333, "end": 40392, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "8026", "dept": "EEE", "start": 21097, "end": 21156, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "8th Floor Open Space", "dept": "CSE", "start": 40393, "end": 40432, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "8th Floor Open Space", "dept": "EEE", "start": 21157, "end": 21196, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "9003", "dept": "CSE", "start": 40433, "end": 40496, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "9003", "dept": "EEE", "start": 21197, "end": 21260, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "9011", "dept": "CSE", "start": 40497, "end": 40576, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "9011", "dept": "EEE", "start": 21261, "end": 21340, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "9016-A", "dept": "CSE", "start": 40577, "end": 40620, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "9016-A", "dept": "EEE", "start": 21341, "end": 21384, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "9016-B", "dept": "CSE", "start": 40621, "end": 40668, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "9016-B", "dept": "EEE", "start": 21385, "end": 21432, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "9017", "dept": "CSE", "start": 40669, "end": 40729, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "9017", "dept": "EEE", "start": 21433, "end": 21493, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "9018", "dept": "CE", "start": 10001, "end": 10066, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "9018", "dept": "EEE", "start": 21494, "end": 21559, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "11002", "dept": "CE", "start": 10067, "end": 10130, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "11002", "dept": "EEE", "start": 21560, "end": 21623, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "11008", "dept": "CE", "start": 10131, "end": 10178, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "11008", "dept": "EEE", "start": 21624, "end": 21671, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "11009", "dept": "CE", "start": 10179, "end": 10226, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "11009", "dept": "EEE", "start": 21672, "end": 21719, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "11010", "dept": "CE", "start": 10227, "end": 10282, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Shahid Syed Nazrul Islam Academic Building (SSNIAB)", "room": "11010", "dept": "EEE", "start": 21720, "end": 21775, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},

    # Building: Textile Workshop Building (TWB)
    {"building": "Textile Workshop Building (TWB)", "room": "Fabric Lab (Ground Floor)", "dept": "CE", "start": 10283, "end": 10322, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "Fabric Lab (Ground Floor)", "dept": "ChE", "start": 70001, "end": 70040, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "Wet Lab (Ground Floor)", "dept": "CE", "start": 10323, "end": 10412, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "Wet Lab (Ground Floor)", "dept": "ChE", "start": 70041, "end": 70129, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "Yarn Lab-2 North Side (Ground Floor)", "dept": "CE", "start": 10413, "end": 10472, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "Yarn Lab-2 North Side (Ground Floor)", "dept": "ChE", "start": 70130, "end": 70188, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "201", "dept": "CE", "start": 10473, "end": 10520, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "201", "dept": "ChE", "start": 70189, "end": 70236, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "216", "dept": "CE", "start": 10521, "end": 10576, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "216", "dept": "ChE", "start": 70237, "end": 70291, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "414", "dept": "CE", "start": 10577, "end": 10606, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "416", "dept": "CE", "start": 10607, "end": 10630, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "417", "dept": "CE", "start": 10631, "end": 10678, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "417", "dept": "TE", "start": 50001, "end": 50048, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "419", "dept": "CE", "start": 10679, "end": 10726, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "419", "dept": "TE", "start": 50049, "end": 50096, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "420", "dept": "CE", "start": 10727, "end": 10774, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "420", "dept": "TE", "start": 50097, "end": 50144, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "511", "dept": "CE", "start": 10775, "end": 10798, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "512", "dept": "CE", "start": 10799, "end": 10854, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "512", "dept": "ChE", "start": 70292, "end": 70347, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "513", "dept": "CE", "start": 10855, "end": 10910, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "513", "dept": "ChE", "start": 70348, "end": 70403, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "516", "dept": "CE", "start": 10911, "end": 10958, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "516", "dept": "ChE", "start": 70404, "end": 70451, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "517", "dept": "CE", "start": 10959, "end": 11018, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "517", "dept": "Arch.", "start": 60001, "end": 60060, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "518", "dept": "CE", "start": 11019, "end": 11066, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "518", "dept": "TE", "start": 50145, "end": 50192, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "519", "dept": "CE", "start": 11067, "end": 11126, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "519", "dept": "TE", "start": 50193, "end": 50252, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "520", "dept": "CE", "start": 11127, "end": 11174, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "520", "dept": "TE", "start": 50253, "end": 50296, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "612", "dept": "CE", "start": 11175, "end": 11230, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "612", "dept": "Arch.", "start": 60061, "end": 60114, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "618", "dept": "CE", "start": 11231, "end": 11254, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "619", "dept": "CE", "start": 11255, "end": 11296, "shift": "1st Shift", "time": "09:30 AM to 12:00 PM"},
    {"building": "Textile Workshop Building (TWB)", "room": "619", "dept": "ChE", "start": 70452, "end": 70493, "shift": "2nd Shift", "time": "02:00 PM to 4:30 PM"},
]

def map_candidate_seat(row):
    try:
        roll = int(row["Applicant's ID"])
    except Exception:
        return pd.Series(["02-08-2026", "Not Found", "Not Found", "Not Found"])
        
    dept_full = row["Department"]
    dept_abbr = dept_mapping.get(dept_full, "")
    
    for entry in seat_plan:
        if entry["dept"] == dept_abbr and entry["start"] <= roll <= entry["end"]:
            shift_with_time = f"{entry['shift']} ({entry['time']})"
            return pd.Series([
                "02-08-2026",
                shift_with_time,
                entry["building"],
                entry["room"]
            ])
            
    return pd.Series(["02-08-2026", "Unknown Shift", "Unknown Building", "Unknown Room"])

# Apply the mapping to the combined dataframe
if 'combined_df' in locals() and not combined_df.empty:
    print("Mapping candidates to their Seat Plan allocations...")
    combined_df[["Date", "Shift with Time", "Building Name", "Room"]] = combined_df.apply(map_candidate_seat, axis=1)
    combined_df["Selected"] = False
    combined_df["Waiting List"] = False
    print("Mapping complete!")
else:
    print("No DataFrame to map.")

Mapping candidates to their Seat Plan allocations...


Mapping complete!


In [4]:
# 4. Display summary and verification of the mapped database
if 'combined_df' in locals() and not combined_df.empty:
    print("--- Combined DataFrame Summary ---")
    print(f"Shape: {combined_df.shape}")
    
    print("\n--- Verification of mapping (Null/Unknown counts) ---")
    print("Null values in Date, Shift, Building, Room:")
    print(combined_df[["Date", "Shift with Time", "Building Name", "Room"]].isnull().sum())
    print("Unmapped count ('Unknown Room'):", (combined_df['Room'] == 'Unknown Room').sum())
    
    print("\n--- Mapped candidates by Building ---")
    print(combined_df['Building Name'].value_counts())
    
    print("\n--- First 5 rows of the mapped database ---")
    display(combined_df.head())
else:
    print("Combined DataFrame is empty or not loaded.")

--- Combined DataFrame Summary ---
Shape: (6583, 15)

--- Verification of mapping (Null/Unknown counts) ---
Null values in Date, Shift, Building, Room:
Date               0
Shift with Time    0
Building Name      0
Room               0
dtype: int64
Unmapped count ('Unknown Room'): 0

--- Mapped candidates by Building ---
Building Name
Shahid Syed Nazrul Islam Academic Building (SSNIAB)    3964
Textile Workshop Building (TWB)                        1917
Shahid Abu Sayed Administrative Building (SASAB)        702
Name: count, dtype: int64

--- First 5 rows of the mapped database ---


,SL,Applicant's ID,Payment ID,Name,Father's Name,Quota,Comment,Department,Source File,Date,Shift with Time,Building Name,Room,Selected,Waiting List
0,1,10001,16737,ANTAR CHANDRA GOSH,GOPAL CHANDRA GOSH,,,Civil Engineering (CE),1 Civil Engineering (CE).pdf,02-08-2026,1st Shift (09:30 AM to 12:00 PM),Shahid Syed Nazrul Islam Academic Building (SS...,9018,False,False
1,2,10002,14215,MD. JESHAN AHMED,SHEHAB UDDIN,,,Civil Engineering (CE),1 Civil Engineering (CE).pdf,02-08-2026,1st Shift (09:30 AM to 12:00 PM),Shahid Syed Nazrul Islam Academic Building (SS...,9018,False,False
2,3,10003,12249,ZAKYA SULTANA,MD. ZOBAYDUR RAHMAN SARKAR,,,Civil Engineering (CE),1 Civil Engineering (CE).pdf,02-08-2026,1st Shift (09:30 AM to 12:00 PM),Shahid Syed Nazrul Islam Academic Building (SS...,9018,False,False
3,4,10004,15118,SHAMSUL,MD. KHADAM ALI,,,Civil Engineering (CE),1 Civil Engineering (CE).pdf,02-08-2026,1st Shift (09:30 AM to 12:00 PM),Shahid Syed Nazrul Islam Academic Building (SS...,9018,False,False
4,5,10005,15463,MD. REYAD HASSAN SAIKAT,MD. SHOHID SHAK,,,Civil Engineering (CE),1 Civil Engineering (CE).pdf,02-08-2026,1st Shift (09:30 AM to 12:00 PM),Shahid Syed Nazrul Islam Academic Building (SS...,9018,False,False


In [5]:
# 5. Export combined and mapped DataFrame to Excel
if 'combined_df' in locals() and not combined_df.empty:
    excel_file = "combined_candidates.xlsx"
    combined_df.to_excel(excel_file, index=False)
    print(f"Successfully exported combined mapped data to '{excel_file}'!")
    print(f"The file contains {len(combined_df)} rows and {len(combined_df.columns)} columns.")
else:
    print("No DataFrame to export.")

Successfully exported combined mapped data to 'combined_candidates.xlsx'!
The file contains 6583 rows and 15 columns.
